# Chapitre 9 - Lab : consensus et coordination avec etcd

etcd est le service de coordination répliqué par **Raft** (cours 5 et 6). Ce lab
lance une grappe de trois nœuds et vérifie sur le vrai système ce que les cours ont
simulé : révisions croissantes et CAS, baux et clés éphémères, verrou dont le bail expire
et jeton de clôture, lectures linéarisables contre lectures sérialisables sur un nœud
isolé, élection d'un nouveau leader après une panne, perte du quorum.

**Démarrer et arrêter la grappe** (depuis la racine du dépôt) :

```bash
docker compose -f docker/consistency/compose.yml -p ddia_consistency up -d
# ... exécuter le notebook ...
docker compose -f docker/consistency/compose.yml -p ddia_consistency down -v
```

Le notebook parle à etcd par sa passerelle JSON/HTTP (ports 11001 à 11003) et pilote Docker
par `subprocess` pour provoquer pannes et coupures réseau (uniquement sur les conteneurs
`ddia_consistency_*`).

**Plan.**

1. La grappe : qui est leader ?
2. Révisions et compare-and-set.
3. Baux et clés éphémères.
4. Un verrou dont le bail expire, et la clôture.
5. Lectures linéarisables et sérialisables sur un nœud isolé.
6. Panne du leader et élection.
7. Perte du quorum.

In [1]:
import base64
import json
import subprocess
import threading
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)

PORTS = {"etcd1": 11001, "etcd2": 11002, "etcd3": 11003}
CONT = {n: f"ddia_consistency_{n}" for n in PORTS}
NET = "ddia_consistency_default"

b64 = lambda s: base64.b64encode(s.encode()).decode()
unb = lambda s: base64.b64decode(s).decode()

def api(node, path, body, timeout=3.0):
    """Appel de la passerelle JSON d'etcd (API v3) sur un nœud."""
    req = urllib.request.Request(f"http://127.0.0.1:{PORTS[node]}/v3/{path}", data=json.dumps(body).encode(), method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def put(node, key, value, lease=None, timeout=3.0):
    body = {"key": b64(key), "value": b64(value)}
    if lease:
        body["lease"] = lease
    return api(node, "kv/put", body, timeout)

def get(node, key, serializable=False, timeout=3.0):
    r = api(node, "kv/range", {"key": b64(key), "serializable": serializable}, timeout)
    kvs = r.get("kvs", [])
    return (unb(kvs[0]["value"]) if kvs else None), r["header"]

def docker(*args):
    return subprocess.run(["docker", *args], capture_output=True, text=True)

def ctl(node, *args):
    """etcdctl exécuté DANS le conteneur (fonctionne même s'il est coupé du réseau)."""
    p = docker("exec", CONT[node], "etcdctl", "--endpoints=127.0.0.1:2379", "--command-timeout=2s", *args)
    return p.returncode, p.stdout.strip(), p.stderr.strip()

def statut():
    rows, ids = [], {}
    for n in PORTS:
        try:
            s = api(n, "maintenance/status", {}, timeout=1.0)
            ids[s["header"]["member_id"]] = n
            rows.append(dict(nœud=n, id=s["header"]["member_id"], leader_id=s.get("leader"), mandat=int(s["raftTerm"]),
                             indice_raft=int(s["raftIndex"]), révision=int(s["header"]["revision"]),
                             erreur=";".join(s.get("errors", []))))
        except Exception as e:
            rows.append(dict(nœud=n, erreur=type(e).__name__))
    df = pd.DataFrame(rows)
    df["leader"] = df.get("leader_id", pd.Series(dtype=object)).map(ids)
    return df.drop(columns=[c for c in ("id", "leader_id") if c in df.columns])

def leader():
    df = statut()
    vals = df["leader"].dropna().unique() if "leader" in df else []
    return vals[0] if len(vals) == 1 else None

def attendre_grappe(timeout=30):
    t0 = time.time()
    while time.time() - t0 < timeout:
        df = statut()
        if "leader" in df and df["leader"].notna().all() and df["leader"].nunique() == 1 and (df["erreur"].fillna("") == "").all():
            return df
        time.sleep(0.3)
    raise RuntimeError("la grappe etcd n'est pas prête : démarrez-la avec docker compose (voir en tête)")

_ = attendre_grappe()

## 1. La grappe : qui est leader ? (section *Membership and Coordination Services*)

Chaque nœud rapporte l'identifiant du leader qu'il connaît, son **mandat** Raft (term),
l'indice de son journal Raft et la **révision** du magasin clé-valeur.

In [2]:
st0 = attendre_grappe()
st0

,nœud,mandat,indice_raft,révision,erreur,leader
0,etcd1,6,897,834,,etcd2
1,etcd2,6,897,834,,etcd2
2,etcd3,6,897,834,,etcd2


Les trois nœuds s'accordent sur un même leader et un même mandat. Le mandat vaut 2 ou plus :
la grappe a élu son premier leader au démarrage (et davantage si elle a déjà subi des
pannes). Toutes les requêtes d'écriture passeront par ce leader, quel que soit le nœud
contacté par le client.

> **À retenir.** etcd est un journal Raft de 3 nœuds : un leader par mandat, et un état
> (révision, indice) que tous les nœuds partagent.

## 2. Révisions et compare-and-set (sections *Implementing linearizable storage using total order broadcast* et *Membership and Coordination Services*)

Chaque écriture incrémente la **révision** globale ; chaque clé garde sa révision de
création (`create_revision`) et de dernière modification (`mod_revision`). Les
**transactions** d'etcd sont des CAS généralisés : « si telle comparaison est vraie, faire
ceci, sinon cela ». Réservons un nom d'utilisateur avec la comparaison
`create_revision == 0` (la clé n'existe pas) : 20 clients concurrents, répartis sur les
trois nœuds, tentent de réserver le même nom.

In [3]:
def reserver(node, nom, user):
    r = api(node, "kv/txn", {"compare": [{"key": b64(nom), "target": "CREATE", "result": "EQUAL", "create_revision": "0"}],
                             "success": [{"request_put": {"key": b64(nom), "value": b64(user)}}]})
    return r.get("succeeded", False), int(r["header"]["revision"])

nom = f"/utilisateurs/bob-{int(time.time())}"
resultats = {}
def tente(i):
    resultats[i] = reserver(list(PORTS)[i % 3], nom, f"client{i}")
threads = [threading.Thread(target=tente, args=(i,)) for i in range(20)]
for t in threads: t.start()
for t in threads: t.join()
gagnants = [i for i, (ok, _) in resultats.items() if ok]
valeur, _ = get("etcd2", nom)
print(f"réservations réussies : {len(gagnants)} (client{gagnants[0]}) ; propriétaire lu sur etcd2 : {valeur}")
revs = []
for k in range(5):
    r = put(list(PORTS)[k % 3], "compteur", str(k))
    revs.append(int(r["header"]["revision"]))
print("révisions de 5 écritures successives (sur 3 nœuds différents) :", revs)

réservations réussies : 1 (client0) ; propriétaire lu sur etcd2 : client0
révisions de 5 écritures successives (sur 3 nœuds différents) : [836, 837, 838, 839, 840]


Une seule des 20 réservations réussit, quel que soit le nœud contacté : le CAS est
linéarisable parce qu'il passe par le journal Raft. Les révisions de cinq écritures
successives, adressées à des nœuds différents, sont strictement croissantes : c'est l'ordre
total du journal partagé, et c'est ce numéro qui sert de **jeton de clôture**.

> **À retenir.** Une transaction etcd est un CAS linéarisable ; la révision est un compteur
> global croissant, utilisable comme identifiant ordonné ou comme jeton.